# Amazon ML Challenge — Business Entity Resolution

This notebook runs the fixed project baseline from GitHub while keeping the dataset, model artifacts, and outputs in Google Drive.

**Approach:** conservative normalization → multi-rule blocking → similarity features → Logistic Regression → macro F0.5 threshold tuning → test prediction.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Configure project locations

Change only `GITHUB_REPO_URL` and, if needed, `DRIVE_PROJECT_DIR`.


In [ ]:
import os

GITHUB_REPO_URL = 'https://github.com/YOUR_USERNAME/amazon-ml-entity-resolution.git'
REPO_DIR = '/content/amazon-ml-entity-resolution'
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/amazon-ml-entity-resolution'

TRAIN_DIR = os.path.join(DRIVE_PROJECT_DIR, 'data', 'train')
TEST_DIR = os.path.join(DRIVE_PROJECT_DIR, 'data', 'test')
ARTIFACTS_DIR = os.path.join(DRIVE_PROJECT_DIR, 'artifacts')
OUTPUT_DIR = os.path.join(DRIVE_PROJECT_DIR, 'output')

for path in [TRAIN_DIR, TEST_DIR, ARTIFACTS_DIR, OUTPUT_DIR]:
    os.makedirs(path, exist_ok=True)

print('Repo:', REPO_DIR)
print('Train:', TRAIN_DIR)
print('Test:', TEST_DIR)
print('Artifacts:', ARTIFACTS_DIR)
print('Output:', OUTPUT_DIR)


In [ ]:
# Clone once; pull latest code on later notebook runs.
import os

if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only

%cd {REPO_DIR}


In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .


## 2. Verify the seven challenge files


In [ ]:
import os

required_files = [
    'train_source1.tsv', 'train_source2.tsv', 'train_source3.tsv', 'train_ground_truth.tsv',
]
required_test_files = [
    'test_source1.tsv', 'test_source2.tsv', 'test_source3.tsv',
]

for filename in required_files:
    path = os.path.join(TRAIN_DIR, filename)
    print(('OK  ' if os.path.exists(path) else 'MISS'), path)

for filename in required_test_files:
    path = os.path.join(TEST_DIR, filename)
    print(('OK  ' if os.path.exists(path) else 'MISS'), path)


In [ ]:
# Optional: inspect the raw files before running the pipeline.
import pandas as pd

s1 = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source1.tsv'), sep='\t', dtype=str).fillna('')
s2 = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source2.tsv'), sep='\t', dtype=str).fillna('')
s3 = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source3.tsv'), sep='\t', dtype=str).fillna('')
gt = pd.read_csv(os.path.join(TRAIN_DIR, 'train_ground_truth.tsv'), sep='\t', dtype=str).fillna('')

print('S1:', s1.shape)
print('S2:', s2.shape)
print('S3:', s3.shape)
print('GT:', gt.shape)
display(s1.head())


## 3. Run EDA


In [ ]:
!python scripts/eda.py --data-dir {TRAIN_DIR}


## 4. Check blocking recall
The blocker uses the **union** of name and address channels. This is intentional: an address match must be able to rescue a completely different name, while a good name must be able to rescue a missing address.

In [ ]:
!python scripts/check_blocking.py --data-dir {TRAIN_DIR}


## 4. Train the baseline and tune the threshold

The model and metadata are written to Drive so they survive the Colab runtime.


In [ ]:
!python scripts/train.py \
    --data-dir {TRAIN_DIR} \
    --artifacts-dir {ARTIFACTS_DIR} \
    --validation-size 0.2 \
    --negatives-per-positive 5 \
    --random-state 42


## 5. Generate test predictions


In [ ]:
!python scripts/predict.py \
    --data-dir {TEST_DIR} \
    --artifacts-dir {ARTIFACTS_DIR} \
    --output-dir {OUTPUT_DIR}


## 6. Validate the two required output files


In [ ]:
!python scripts/validate.py \
    --matching {OUTPUT_DIR}/matching_results.tsv \
    --candidate {OUTPUT_DIR}/candidate_pairs.tsv \
    --test-dir {TEST_DIR}


## 7. Inspect outputs


In [ ]:
display(pd.read_csv(os.path.join(OUTPUT_DIR, 'matching_results.tsv'), sep='\t').head(10))
display(pd.read_csv(os.path.join(OUTPUT_DIR, 'candidate_pairs.tsv'), sep='\t').head(10))


## 8. Package the final submission

Run this only after the output validator passes and the methodology document is complete.


In [ ]:
TEAM_NAME = 'YOUR_TEAM_NAME'
!python scripts/package_submission.py --team-name {TEAM_NAME} --documentation Documentation_template.md --output {OUTPUT_DIR}/{TEAM_NAME}_submission.zip
